# SECOM Data Modeling
---

### Preprocessing the data

In [ ]:
# Imports
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    average_precision_score,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

In [ ]:
# Pulling in the SECOM data and loading data into feature and label dataframes
secom = fetch_ucirepo(id=179)
df = pd.DataFrame(secom.data.original)
X = df.drop(columns=["class", "timestamp"])
y = df["class"]
# converting "-1" passing label to "0"
y = y.replace(-1, 0)

In [ ]:
# filtering out features with null percentage above threshold and imputing NaN with median
threshold = 90 # null pct threshold

null_cols = X.columns[X.isna().mean()*100 > threshold]
X_filtered = X.drop(columns=null_cols)
X_filtered = X_filtered.fillna(X_filtered.median())

In [ ]:
# Remove constant variance features
zero_var_cols = X_filtered.columns[X_filtered.var() == 0]
X_filtered = X_filtered.drop(columns=zero_var_cols)

In [ ]:
# dropping columns correlated above threshold
corr = X_filtered.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

to_drop = [
    col
    for col in upper.columns
    if any(upper[col] > 0.95)
]

X_uncorr = X_filtered.drop(columns=to_drop)
X_uncorr.shape

___

### Test-train split and scaling

In [ ]:
# Splitting data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_uncorr, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Scaling the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

In [ ]:
# train the logistic regression classifier
lr_model = LogisticRegression(penalty='l1', random_state=42, class_weight="balanced", solver='liblinear', max_iter=5000)
lr_model.fit(X_train_scaled, y_train)
y_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# Sweep threshold probability and print classification report
for threshold in [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]:
    
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\nThreshold = {threshold}")

    print(
        classification_report(
            y_test,
            y_pred,
            digits=3
        )
    )

In [ ]:
# Print performance metrics
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))

In [ ]:
# Plot the confusion matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["Pass", "Fail"])
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Plot ROC curve
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve")
plt.show()

In [ ]:
# Plot precision-recall curve
PrecisionRecallDisplay.from_predictions(y_test, y_prob)
plt.title("Precision-Recall Curve")
plt.show()

In [ ]:
# Checking coefficients
coef = pd.Series(
    lr_model.coef_[0],
    index=X_train.columns
)
selected_features = coef[coef != 0].sort_values(key=abs, ascending=False)
print("Number of selected features:", selected_features.shape[0])
print(selected_features.head(30))

In [ ]:
# Plotting top coefficients
top_coef = selected_features.head(20).sort_values()

top_coef.plot(kind="barh", figsize=(8, 6))

plt.xlabel("Coefficient")
plt.title("Top L1 Logistic Regression Coefficients")
plt.show()

## Learnings from LR
1. We tried several different probaility thresholds for prediction, failure class precision is poor and largely unaffected by threshold
2. Even with L1 regularization, we still have ~220 features with non-zero coefficients
    - indicative that importance is spread over sensors or there are interactions present which need non-linear modeling